# k-means|| vs serial k-means++ vs Random — Comparison (todo #4)

Notebook dedicated to the comparison between k-means|| and the two serial baselines used
in the paper by Bahmani et al. (*Scalable K-Means++*, VLDB 2012): classic k-means++
and Random initialization. Same 10% of KDDCup99 already used in
`analysis.ipynb`, to stay directly comparable with that notebook.

Modules used:
- `kmeans_parallel.py` — distributed k-means|| algorithm (unchanged)
- `kmeans_serial.py` — serial k-means++ and Random (new)
- `kmeans_comparison.py` — three-way comparison driver (new)
- `comparison_analysis.py` — tables and plots dedicated to the comparison (new)
- `launch_cluster.py`, `data_loader.py`, `benchmark.py` — unchanged, reused as they are

## 1. Import

In [ ]:
import time
import numpy as np
import pandas as pd

from src.kmeans_parallel import kmeans_parallel
from src.kmeans_serial import kmeans_serial
from src.kmeans_comparison import run_comparison, pilot_timing_check, _materialize_bag
from src.comparison_analysis import summarize_comparison, format_paper_table, plot_cost_by_method, plot_cost_vs_rounds

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset
from src.benchmark import calculate_inertia


## 2. Configurable parameters

Cluster and dataset: same configuration as `analysis.ipynb`, to work on the
same standardized data. `MAX_ITER_FIT`/`TOL` are more generous than the
default used in the partitioning sweeps (`max_iter_fit=10`),
because here we want the cost *at convergence* for all three methods, not
just a fixed budget to compare speeds.

In [ ]:
# --- Cluster ---
N_WORKERS = 8      # between 1 and 8 (nodes available in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # empirical rule: >= n_threads_per_worker * n_workers

# --- Comparison ---
MAX_ITER_FIT = 100          # generous budget: we want the cost at convergence, not just fast
TOL = 1e-4
SEED = 42
AVERAGING_ITERATIONS = 11   # paper convention: median over 11 runs for the cost tables
N_LOCAL_TRIALS = None       # None = greedy k-means++ (sklearn default); 1 = "vanilla" Algorithm 1 of the paper

In [ ]:
# --- Dataset (identical to analysis.ipynb, section 2) ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = '/tmp/kddcup_data_shards'  # directory of Parquet shard files on master

## 3. Avvio/connessione al cluster

In [ ]:
# DO NOT RUN if already existing!
cluster, client = launch_cluster(N_WORKERS)


#### IF INSTEAD ALREADY EXISTING CLUSTER:

In [ ]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")


In [ ]:
client.scheduler_info()


## 4. Dataset loading (same 10% KDDCup99 as `analysis.ipynb`)

In [ ]:
start = time.time()
X_bag, (mean_ar, std_ar) = load_dataset(
    n_partitions=NUM_PARTITIONS,
    client=client,
    dataset_url=DATASET_URL_10PC,
    raw_gz_path=RAW_GZ_PATH,
    parquet_path=PARQUET_PATH,
    col_names=COL_NAMES,
)
elapsed = time.time() - start
print(f"Time elapsed: {elapsed:.2f} s")


## 5. Feasibility check

The *greedy* k-means++ seeding (sklearn default) on ~494k rows can
be slow. Before launching the full comparison loop, time
a single run (seed + fit) at the smallest `k` you intend to
test below.

In [ ]:
K_SMALLEST = 500  # must match the smallest value in K_VALUES (section 6)

X_arr = _materialize_bag(client, X_bag)

for init in ("k-means++", "random"):
    pilot_timing_check(
        X_arr, k=K_SMALLEST, init=init, seed=SEED,
        n_local_trials=N_LOCAL_TRIALS, max_iter=MAX_ITER_FIT, tol=TOL,
    )

**Decision point**: multiply the total time by
`AVERAGING_ITERATIONS` and by the number of values in `K_VALUES`. If it is
not acceptable, in order of preference:

1. reduce `AVERAGING_ITERATIONS` only for the serial side (in the code of
   `kmeans_comparison.run_comparison` — a small modification is required
   if you want different repetitions between k-means|| and the serial baselines);
2. as a last resort, subsample further **both** `X_arr` and
   `X_bag` used in the comparison — otherwise the costs are no longer
   comparable in absolute value between the three methods.

## 6. Comparison configuration

`K_VALUES` and `parallel_combinations` must be filled with your choices.
If in `parallel_combinations` you include several values of `r` at the same
`l_over_k`, Section 8 can also reproduce the cost-vs-rounds plot in the
style of Figure 5.2/5.3 of the paper.

In [ ]:
K_VALUES = [500, 1000]  # <- choose here the k values to test

parallel_combinations = [
    # (num_partitions, l_over_k, r) — choose here the k-means|| combinations to compare
    (NUM_PARTITIONS, 1.0, 5),
    (NUM_PARTITIONS, 2.0, 5),
]

## 7. Running the comparison

In [ ]:
df_comparison = run_comparison(
    client, X_bag,
    k_values=K_VALUES,
    parallel_combinations=parallel_combinations,
    seed=SEED,
    averaging_iterations=AVERAGING_ITERATIONS,
    max_iter_fit=MAX_ITER_FIT,
    tol=TOL,
    n_local_trials=N_LOCAL_TRIALS,
    label="kmeans_comparison",
)
df_comparison


## 8. Analysis

Table in the style of Tables 1/2 of the paper (median over `AVERAGING_ITERATIONS`
runs, seed/final side by side), mean±std details, and a bar chart to compare
the categories (the three methods) per `k`.

In [ ]:
print(format_paper_table(df_comparison, stat="median"))


In [ ]:
summarize_comparison(df_comparison)


In [ ]:
plot_cost_by_method(df_comparison, metric="cost_seed")
plot_cost_by_method(df_comparison, metric="cost_final")


Cost-vs-rounds plot (Figure 5.2/5.3-style) — only if in
`parallel_combinations` you included several values of `r` at the same `l_over_k`
for one of the `K_VALUES`:

In [ ]:
# plot_cost_vs_rounds(df_comparison, k=500, metric="cost_final")


**Note on times**: `time_seed`/`time_fit` in `df_comparison` are
reported only for reference, not as a controlled comparison —
`kmeans_serial` runs single-threaded on the client process, `kmeans_parallel`
runs distributed on the Dask cluster. For a correct wall-clock comparison
comparable hardware would be needed (e.g. a single cluster node), which is
out of scope for this notebook. The primary comparison axis here is the
quality (`cost_seed`/`cost_final`), as in the paper.

## 9. Cluster shutdown

Run at the end of the work, or before relaunching `launch_cluster` with a
different `N_WORKERS`.

In [ ]:
shutdown_cluster(cluster, client)
